# 01 · Data acquisition

**DSCI 510 project: Anticipating Attacks on Software Flaws**

This notebook downloads everything the analysis needs and builds the tables that the later notebooks
start from. It only uses methods that the validation notebook (`00_validate_sources.ipynb`) tested
against the live services on November 10, 2026.

| Stage | What it does | Main output |
|:--|:--|:--|
| **A. Setup** | Settings, the API key, the HTTP helper, the NVD client and the parser | (functions only) |
| **B. NVD download** | Every CVE published 2016–2024, one file per 119-day window | `data/interim/nvd_2016_2024.parquet` |
| **C. KEV snapshot** | A dated copy of CISA's Known Exploited Vulnerabilities catalogue, pinned to one version | `data/raw/kev/kev_<version>.parquet` |
| **D. Analysis table** | One row per live CVE: publisher-only fields, the 365-day label and the split | `data/processed/analysis_table.parquet` |
| **E. EPSS benchmark** | Each 2024 CVE's EPSS score 1, 7 and 30 days after publication | `data/processed/epss_benchmark.parquet` |
| **F. Summary** | Fingerprints of the tables, every check, the files written and a run log | `logs/acquisition_log.json` |

**How to run:** keep the `.env` file with `NVD_API_KEY=...` beside this notebook, then choose
**Kernel → Restart Kernel and Run All Cells**.

**Runtime:** about 45–60 minutes in total. Most of it is deliberate waiting between requests: about
20–30 minutes for the NVD and 15–25 minutes for the EPSS files.

**If the run stops part-way:** run it again. Every finished window and day is saved as it completes
and skipped on the next run, so a re-run only fetches what is missing.

**Reproducibility:** the data are fixed once downloaded. A re-run in the same folder reuses the saved
NVD windows and EPSS days, and loads the **pinned** KEV snapshot (`KEV_SNAPSHOT` in Step 2) instead of
downloading today's catalogue, so it rebuilds exactly the same tables in about a minute. Step 20
proves it: it fingerprints every table and compares the fingerprints with the previous run's.

**Checks:** as in the validation notebook, every check prints `PASS`, `WARN` (it ran, but a person
should read the result) or `FAIL`, and the notebook keeps going. Stage F lists them all.

---

## Stage A · Setup

### Step 1 · Libraries

**What we are doing:** installing the four libraries the notebook needs into the kernel that runs it,
then importing them and checking their versions.

**Why:** the notebook should run on a machine that has never run it before. `%pip` installs into the
running kernel, which is not always the Python found first on the system path. `pyarrow` is required
because the results are saved as parquet files, which keep dates and data types that a CSV file would
turn back into text.

In [1]:
%pip install --quiet requests pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.


**Imports and version check:** the next cell imports everything the notebook uses and stops with a clear message if `pyarrow` is missing or pandas is older than 2.0.

In [2]:
import os, sys, io, json, gzip, time, shutil, hashlib
import datetime as dt
from collections import Counter
from pathlib import Path
from getpass import getpass
from urllib.parse import urlencode

import requests
import pandas as pd
import numpy as np

try:
    import pyarrow  # used by pandas to read and write parquet files
except ImportError as exc:
    raise RuntimeError("pyarrow is required: run the %pip cell above, restart the kernel, "
                       "and run the notebook again.") from exc

# pandas 2.0+ is needed for format="ISO8601" when parsing NVD's timestamps.
if int(pd.__version__.split(".")[0]) < 2:
    raise RuntimeError(f"pandas {pd.__version__} is too old: pandas 2.0 or newer is required.")

VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
            "requests": requests.__version__, "pyarrow": pyarrow.__version__}
print(" | ".join(f"{name} {version}" for name, version in VERSIONS.items()))

python 3.14.6 | pandas 3.0.3 | numpy 2.4.6 | requests 2.34.2 | pyarrow 23.0.1


**Reading the result:** one line of library versions. A "restart the kernel" note after the `%pip`
cell is normal the first time; if the import cell raises an error, restart the kernel and run all
cells again.

### Step 2 · Settings

**What we are doing:** collecting every setting in one place: where files go, the study design, and
how politely to pace requests.

**Why:** a setting that appears once, at the top, can be checked and changed without searching the
code. The study design below is the one fixed in the proposal and confirmed by the validation run.
`KEV_SNAPSHOT` pins the version of the KEV catalogue the labels are built from (Step 12).

In [3]:
# ---- Where files go (all beside this notebook; data/ and logs/ are excluded from git) ----
DATA      = Path("data")
RAW_NVD   = DATA / "raw" / "nvd"          # per window: raw records (.jsonl.gz) and flat rows (.parquet)
RAW_KEV   = DATA / "raw" / "kev"          # dated KEV snapshots
RAW_EPSS  = DATA / "raw" / "epss_daily"   # per day: only the EPSS rows the benchmark needs
INTERIM   = DATA / "interim"              # all NVD records in one table
PROCESSED = DATA / "processed"            # the tables the later notebooks start from
LOGS      = Path("logs")
for folder in (RAW_NVD, RAW_KEV, RAW_EPSS, INTERIM, PROCESSED, LOGS):
    folder.mkdir(parents=True, exist_ok=True)

# ---- Study design (proposal, confirmed by the validation run) ----
FIRST_YEAR, LAST_YEAR = 2016, 2024
KEV_LAUNCH = dt.datetime(2021, 11, 3)     # the KEV catalogue's first day
LABEL_DAYS = 365                          # exploited = listed in KEV within 365 calendar days
SPLITS = {                                # [start, end) on NVD's publication date
    "pre-KEV":  (dt.datetime(2016, 1, 1), KEV_LAUNCH),          # description only: no fair label
    "train":    (KEV_LAUNCH, dt.datetime(2023, 1, 1)),
    "validate": (dt.datetime(2023, 1, 1), dt.datetime(2024, 1, 1)),
    "test":     (dt.datetime(2024, 1, 1), dt.datetime(2025, 1, 1)),
}
EPSS_SPLIT = "test"                       # the EPSS benchmark is collected for the test year only
EPSS_HORIZONS = (1, 7, 30)                # days after publication
EPSS_SLACK_DAYS = 2                       # a score missing on its day may be taken up to 2 days later

# ---- Reproducibility ----
KEV_SNAPSHOT = "2026.11.10"               # pinned KEV version: reuse its saved copy (None = download today's)

# ---- Pacing and robustness ----
NVD_PAUSE  = 6.0                          # seconds between NVD requests (NIST asks bulk users to pace)
EPSS_PAUSE = 0.5                          # seconds between EPSS file downloads
NVD_ATTEMPTS_PER_WINDOW = 2               # a window that arrives incomplete is downloaded once more
DRIFT_TOLERANCE = 0.005                   # counts within 0.5% of the validation run: WARN, not FAIL

free_gb = shutil.disk_usage(".").free / 1e9
print(f"settings ready | {len(SPLITS)} splits | EPSS horizons {EPSS_HORIZONS} | free disk {free_gb:,.0f} GB")

settings ready | 4 splits | EPSS horizons (1, 7, 30) | free disk 226 GB


**Reading the result:** one line confirming the settings and the free disk space. The run writes
roughly 300 MB in total (mostly the compressed raw NVD records), so anything above a few GB is plenty.

### Step 3 · Recording checks, and the targets from the validation run

**What we are doing:** defining the small functions every later check uses, and loading the figures
that the latest validation run measured.

**Why:** each check records a verdict in one place, so Stage F can list them all and save them in a
run log. The validation report says exactly how many records and exploited CVEs to expect, so this
download can be checked against those numbers instead of against guesses. If no validation report is
found, the comparisons are reported as `WARN` rather than skipped silently.

In [4]:
def now_iso():
    "Current local time with its UTC offset, e.g. 2026-11-10T09:15:02-08:00."
    return dt.datetime.now().astimezone().isoformat(timespec="seconds")

RUN = {"started": now_iso(), "checks": {}, "sources": {}, "files": {}, "versions": VERSIONS}

def check(name, ok, **details):
    "Record one check and print its verdict. ok=True is PASS, False is FAIL, None is WARN."
    verdict = "WARN" if ok is None else ("PASS" if ok else "FAIL")
    RUN["checks"][name] = {"verdict": verdict, **details}
    print(f"{verdict}  {name}" + (f"  |  {details}" if details else ""))

def show_conditions(conditions):
    "Print a {description: passed} checklist; return the descriptions that failed."
    for description, passed in conditions.items():
        print(f"  {'ok ' if passed else 'NO '} {description}")
    return [description for description, passed in conditions.items() if not passed]

def note_source(name, **details):
    "Stamp a source's access time at the moment its data arrived (a graded requirement)."
    RUN["sources"][name] = {"accessed": now_iso(), **details}

def compare_to_expected(actual, expected, tolerance=DRIFT_TOLERANCE):
    "True if equal, None (WARN) if within the tolerance or no target exists, False otherwise."
    if expected is None:
        return None
    if actual == expected:
        return True
    return None if expected and abs(actual - expected) / expected <= tolerance else False

The cell above defines `check()`, which every later step uses to record a verdict, and `compare_to_expected()`, which turns a count into `PASS`, `WARN` or `FAIL` against a target.

**Loading the targets:** the next cell reads the newest dated validation report in `logs/` and keeps the counts this download should reproduce.

In [5]:
def load_expectations(logs=LOGS):
    "The figures measured by the newest dated validation report, or None if there is none."
    reports = sorted(logs.glob("validation_report_*.json"))
    if not reports:
        print("no dated validation report in logs/: count checks will be WARN")
        return None
    report = json.loads(reports[-1].read_text(encoding="utf-8"))
    rename = {"pre-KEV (description only)": "pre-KEV"}            # the validation's split label
    try:
        sizes = report["findings"]["dataset_size"]
        labels = report["findings"]["label_feasibility"]["splits"]
        return {
            "report": reports[-1].name,
            "total_records": report["steps"]["17-dataset-size"]["total_2016_2024"],
            "live_records": report["steps"]["17-dataset-size"]["excluding_rejected"],
            "live_by_split": {rename.get(r["split"], r["split"]): r["excluding_rejected"] for r in sizes},
            "positives_by_split": {rename.get(r["split"], r["split"]): r["listed_within_365d"] for r in labels},
            "kev_launch_day_entries": report["steps"]["12-kev"]["launch_day_entries"],
        }
    except (KeyError, TypeError) as exc:
        print(f"{reports[-1].name} lacks an expected field ({exc!r}): count checks will be WARN")
        return None

EXPECTED = load_expectations()
if EXPECTED:
    print(f"targets from {EXPECTED['report']}:")
    print(f"  records 2016-2024: {EXPECTED['total_records']:,} ({EXPECTED['live_records']:,} live)")
    print(f"  live records by split: {EXPECTED['live_by_split']}")
    print(f"  exploited within {LABEL_DAYS} days, by split: {EXPECTED['positives_by_split']}")

targets from validation_report_2026-11-10.json:
  records 2016-2024: 200,978 (187,219 live)
  live records by split: {'pre-KEV': 90082, 'train': 28368, 'validate': 28816, 'test': 39953}
  exploited within 365 days, by split: {'pre-KEV': 159, 'train': 128, 'validate': 145, 'test': 149}


**Reading the result:** the targets should read 200,978 records (187,219 live) and 128 / 145 / 149
exploited CVEs in the train, validate and test periods, as measured on November 10.

### Step 4 · Load the API key without displaying it

**What we are doing:** reading the NVD API key from an environment variable, else from the `.env`
file, else from a hidden prompt.

**Why:** every NVD request uses the key, and a notebook saves its printed output, so a key printed even
once would be submitted with the code. Only its length is shown.

In [6]:
def load_api_key(var="NVD_API_KEY", env_file=".env"):
    "Return (key, where it came from): environment variable, then .env file, then a hidden prompt."
    if os.environ.get(var):
        return os.environ[var].strip(), "environment variable"
    path = Path(env_file)
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            name, sep, value = line.strip().partition("=")
            if sep and name.strip() == var:
                value = value.strip().strip('"').strip("'")
                if value and "your-key" not in value:          # ignore the README's placeholder
                    return value, f"{env_file} file"
    return getpass(f"{var} (hidden input): ").strip(), "interactive prompt"

NVD_API_KEY, key_source = load_api_key()
check("A1-key-loaded", len(NVD_API_KEY) > 20, source=key_source, length=len(NVD_API_KEY))

PASS  A1-key-loaded  |  {'source': '.env file', 'length': 36}


**Reading the result:** `PASS` with `length: 36` (NVD keys are 36-character UUIDs). A `FAIL` means
the `.env` file is missing or still holds the placeholder.

### Step 5 · One HTTP helper for every service

**What we are doing:** defining `http_get`, the single function through which every request in this
notebook passes. It is the helper that passed validation, unchanged.

**Why:** retries and error handling belong in one place, so every source is treated the same way. When
a server throttles us or has a temporary fault, the helper waits 6, 12 and then 24 seconds before
giving up. It turns network failures into readable messages, and raises `ApiError` so that later
cells can catch our failures without hiding genuine bugs. The NVD explains a refusal in a `message`
header with an empty body, so that header is read first.

In [7]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "DSCI510-course-project/1.0 (academic research)",
                        "Accept": "application/json"})

class ApiError(RuntimeError):
    "A failure we understand. `status` holds the HTTP status code when there was one."
    def __init__(self, message, status=None):
        super().__init__(message)
        self.status = status

RETRYABLE = {403, 429, 500, 502, 503, 504}        # throttling, or a server that may recover

def reason(resp):
    "NVD explains refusals in a `message` header with an empty body; other servers use the body."
    return resp.headers.get("message") or resp.text[:200].strip() or "(no reason given)"

def query(params=None, flags=()):
    "Build a query string; NVD flags such as noRejected take no value, so they are appended by hand."
    parts = [urlencode(params or {})] + list(flags)
    return "&".join(part for part in parts if part)

The cell above sets up a shared session (one connection reused for every request), the `ApiError` exception and the list of status codes worth retrying.

**The request function itself:** the next cell defines `http_get`, which retries throttled or failed requests with increasing waits and pauses after every successful one.

In [8]:
def http_get(url, params=None, headers=None, max_attempts=4, pause=1.0, timeout=60):
    "GET with backoff. Returns the response, or raises ApiError with a readable cause."
    last = None
    for attempt in range(1, max_attempts + 1):
        final = attempt == max_attempts
        try:
            resp = SESSION.get(url, params=params, headers=headers, timeout=timeout)
        except requests.exceptions.SSLError as exc:
            raise ApiError("TLS failure: usually a VPN or proxy intercepting HTTPS.") from exc
        except requests.exceptions.Timeout:
            last = f"no response within {timeout}s"          # worth another try
            if not final:
                time.sleep(2 ** attempt)
            continue
        except requests.exceptions.ConnectionError as exc:
            raise ApiError(f"cannot reach {url}: check the connection, a firewall or a VPN.") from exc
        if resp.status_code == 200:
            time.sleep(pause)                                # pace the next request
            return resp
        if resp.status_code in RETRYABLE:
            last = f"HTTP {resp.status_code}: {reason(resp)}"
            if not final:
                wait = 6 * 2 ** (attempt - 1)                # 6 s, 12 s, 24 s
                print(f"  {last}; waiting {wait}s (attempt {attempt}/{max_attempts})")
                time.sleep(wait)
            continue
        raise ApiError(f"HTTP {resp.status_code} from {url}: {reason(resp)}", status=resp.status_code)
    raise ApiError(f"gave up after {max_attempts} attempts; last problem: {last}")

print("HTTP helper ready")

HTTP helper ready


**Reading the result:** `HTTP helper ready`. Nothing has been requested yet.

### Step 6 · The NVD client

**What we are doing:** defining `NVDClient`, the class that talks to the NVD CVE API, and
`check_windows`, which verifies a list of date windows. Both passed validation (steps 09 and 15).

**Why:** the NVD returns at most 2,000 records per page and refuses date ranges longer than 120 days,
so 2016–2024 must be fetched as a series of windows, each in several pages.

- `iter_cves` is a generator: it fetches one page, hands the records over one at a time, and at the
  end compares the number received with the total the NVD reported (`last_run`).
- `windows_between` splits a date range into windows of 119 days, one day inside the limit.
- Windows include their start and exclude their end (validation step 15), so neighbouring windows
  neither miss nor repeat a record.

In [9]:
NVD_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

class NVDClient:
    "NVD CVE API 2.0: rate-limited, retrying, paginating and self-reconciling."
    PAGE = 2000                                   # the API's maximum page size
    MAX_WINDOW_DAYS = 120                         # the API's maximum date range
    FMT = "%Y-%m-%dT%H:%M:%S.000Z"

    def __init__(self, api_key, pause=NVD_PAUSE, timeout=120):
        self.headers = {"apiKey": api_key}        # a header, never a query parameter
        self.pause, self.timeout = pause, timeout
        self.last_run = {}

    def get(self, params=None, flags=()):
        "One request; returns the parsed JSON page."
        return http_get(NVD_URL, params=query(params, flags), headers=self.headers,
                        pause=self.pause, timeout=self.timeout).json()

    def iter_cves(self, flags=(), **params):
        "Yield every CVE matching a query, page by page; then record expected vs received."
        start, expected, received = 0, None, 0
        while expected is None or start < expected:
            page = self.get({**params, "resultsPerPage": self.PAGE, "startIndex": start}, flags)
            total = page["totalResults"]
            if expected is None:
                expected = total
            elif total != expected:
                print(f"  note: NVD's total changed mid-window ({expected:,} -> {total:,})")
            batch = page["vulnerabilities"]
            if not batch:
                break                             # an empty page: stop, and let the count tell
            for item in batch:
                yield item["cve"]
            received += len(batch)
            start += len(batch)
        self.last_run = {"expected": expected, "received": received, "complete": received == expected}

The cell above defines the client. Nothing has been requested yet.

**The date windows:** the next cell splits 2016–2024 into windows the NVD will accept, and checks the list for gaps, overlaps and windows that are too wide.

In [10]:
def windows_between(start, stop, max_days=NVDClient.MAX_WINDOW_DAYS):
    "Yield (start, end) ISO strings covering [start, stop), each at most max_days - 1 days wide."
    step, cursor = dt.timedelta(days=max_days - 1), start
    while cursor < stop:
        end = min(cursor + step, stop)
        yield cursor.strftime(NVDClient.FMT), end.strftime(NVDClient.FMT)
        cursor = end

def check_windows(pairs, start, stop, max_days=NVDClient.MAX_WINDOW_DAYS):
    "Return the problems with a window list: too wide, gaps or overlaps, or wrong ends."
    parse = lambda s: dt.datetime.strptime(s, NVDClient.FMT)
    problems = []
    if max(parse(b) - parse(a) for a, b in pairs) > dt.timedelta(days=max_days):
        problems.append(f"a window is wider than {max_days} days")
    if any(pairs[i][0] != pairs[i - 1][1] for i in range(1, len(pairs))):
        problems.append("a gap or overlap between windows")
    if parse(pairs[0][0]) != start or parse(pairs[-1][1]) != stop:
        problems.append("the windows don't start and end on the requested dates")
    return problems

SPAN = (dt.datetime(FIRST_YEAR, 1, 1), dt.datetime(LAST_YEAR + 1, 1, 1))
WINDOWS = list(windows_between(*SPAN))
problems = check_windows(WINDOWS, *SPAN)
check("A2-windows", not problems, windows=len(WINDOWS), problems=problems)

PASS  A2-windows  |  {'windows': 28, 'problems': []}


**Reading the result:** `PASS` with `windows: 28`, the same 28 windows the validation counted.

### Step 7 · The parser: one nested record to one flat row

**What we are doing:** defining `flatten()`, which turns one deeply nested NVD record into one flat
row. This is the parser that passed validation step 10, unchanged.

**Why:** this is where the project's central rule, *only what was known at publication*, becomes code:

- Severity (CVSS) and weakness class (CWE) are taken **only from the CVE's own CNA**, the organisation
  that published it. Every CVSS version is searched, so a CNA that scored only in version 4.0 still
  counts.
- If the CNA gave no score, `cvss_missing` is `True`; the score is never borrowed from NVD's analysts
  or CISA-ADP, who add theirs later.
- Values added after publication, or describing NVD's own workflow, are kept in columns starting
  with `diag_`. They are for diagnosis only and must never become model inputs.

In [11]:
ADP_CISA = "134c704f-9b21-4f2e-91b3-4a467353bcc0"     # CISA's Authorized Data Publisher (not a CNA)
CVSS_VERSIONS = [("cvssMetricV31", "3.1"), ("cvssMetricV30", "3.0"),
                 ("cvssMetricV40", "4.0"), ("cvssMetricV2", "2.0")]

def source_kind(source, cna):
    "Who produced a score or CWE: the CVE's own CNA, NVD's analysts, CISA-ADP, or another party."
    if not source:
        return None
    if source == cna:
        return "CNA"
    return {"nvd@nist.gov": "NVD", ADP_CISA: "CISA-ADP"}.get(source, "other")

def english(cve):
    "The English description (a record can carry several languages)."
    return next((d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"), "")

def to_naive_utc(series):
    "Parse ISO timestamps (with or without a trailing Z) to naive UTC; unreadable values become NaT."
    return pd.to_datetime(series, errors="coerce", format="ISO8601", utc=True).dt.tz_localize(None)

def calendar_days(later, earlier):
    "Whole calendar days between two date columns: a same-day pair is 0, never -1."
    return (later.dt.normalize() - earlier.dt.normalize()).dt.days

The cell above defines the small helpers the parser relies on: who supplied a value, the English description, and how dates are read and subtracted.

**Finding the CNA's own values:** the next cell defines `first_score`, which looks for a score from one particular supplier across every CVSS version, and `weakness_lists`, which separates the CNA's weakness classes from those added by others.

In [12]:
def first_score(metrics, cna, kind):
    "The first score supplied by `kind` (CNA, NVD, CISA-ADP), searching every CVSS version in order."
    for key, label in CVSS_VERSIONS:
        for m in metrics.get(key, []):
            if source_kind(m.get("source"), cna) == kind:
                data = m.get("cvssData", {})
                return (data.get("baseScore"), data.get("vectorString"),
                        data.get("baseSeverity") or m.get("baseSeverity"), label)
    return (None, None, None, None)

def weakness_lists(cve, cna):
    "CNA CWEs in the CNA's order; other parties' CWEs; NVD placeholders; and each entry's who:type."
    cwe_cna, cwe_other, placeholders, entry_types = [], [], [], []
    for w in cve.get("weaknesses", []):
        kind = source_kind(w.get("source"), cna)
        entry_types.append(f"{kind}:{w.get('type')}")
        for d in w.get("description", []):
            value = str(d.get("value", ""))
            if value.startswith("CWE-"):
                target = cwe_cna if kind == "CNA" else cwe_other
                if value not in target:
                    target.append(value)
            elif value:
                placeholders.append(f"{value}:{kind}")          # e.g. NVD-CWE-noinfo
    return cwe_cna, cwe_other, placeholders, entry_types

**Putting it together:** the next cell defines `flatten`, which uses the helpers above to build one flat row. Every column whose name starts with `diag_` is kept for diagnosis only.

In [13]:
def flatten(cve):
    "One NVD record -> one flat row. Model inputs may use only what the CVE's own CNA supplied."
    cna, metrics = cve.get("sourceIdentifier"), cve.get("metrics", {})
    score, vector, severity, version = first_score(metrics, cna, "CNA")
    nvd_score, _, _, nvd_version = first_score(metrics, cna, "NVD")
    adp_score, _, _, adp_version = first_score(metrics, cna, "CISA-ADP")
    cwe_cna, cwe_other, placeholders, entry_types = weakness_lists(cve, cna)
    refs, text = cve.get("references", []), english(cve)
    return {
        "cve_id": cve.get("id"),
        "published": cve.get("published"),
        "rejected": cve.get("vulnStatus") == "Rejected",           # a filter, not a feature
        "cna": cna,
        "description": text,
        "description_len": len(text),
        "cvss_score": score, "cvss_severity": severity,            # CNA only
        "cvss_vector": vector, "cvss_version": version,
        "cvss_missing": score is None,                             # the "severity missing" flag
        "cwes": ";".join(cwe_cna),
        "cwe_primary": cwe_cna[0] if cwe_cna else None,
        "cwe_missing": not cwe_cna,
        # ---- diagnostics only: added after publication, or NVD's workflow. Never model inputs. ----
        "diag_vuln_status": cve.get("vulnStatus"),
        "diag_last_modified": cve.get("lastModified"),
        "diag_nvd_score": nvd_score, "diag_nvd_version": nvd_version,
        "diag_adp_score": adp_score, "diag_adp_version": adp_version,
        "diag_score_sources": ";".join(sorted({f"{label}:{source_kind(m.get('source'), cna)}"
                                              for key, label in CVSS_VERSIONS for m in metrics.get(key, [])})),
        "diag_cwes_non_cna": ";".join(cwe_other),
        "diag_cwe_placeholders": ";".join(placeholders),
        "diag_cwe_entry_types": ";".join(entry_types),
        "diag_n_references": len(refs),
        "diag_ref_tags": ";".join(sorted({t for r in refs for t in r.get("tags", [])})),
        "diag_n_configurations": len(cve.get("configurations", [])),
        "diag_has_nvd_kev_fields": "cisaExploitAdd" in cve,
    }

**What we are doing next:** testing the parser on two hand-made records whose right answers are known,
before trusting it with 200,000 real ones.

**Why:** a parser that quietly took NVD's later score when the CNA had none would train the model on
information a defender did not have on day one. Hand-made records make each rule checkable without
depending on what happens to be in the live data.

In [14]:
def _metric(source, score):
    return {"source": source, "type": "Secondary",
            "cvssData": {"baseScore": score, "vectorString": "AV:N", "baseSeverity": "HIGH"}}

_cna = "psirt@vendor.example"
_full = {"id": "CVE-0000-0001", "sourceIdentifier": _cna, "published": "2023-01-10T18:15:00.000",
         "vulnStatus": "Modified",
         "descriptions": [{"lang": "es", "value": "texto"}, {"lang": "en", "value": "English text"}],
         "metrics": {"cvssMetricV31": [_metric("nvd@nist.gov", 9.8), _metric(ADP_CISA, 9.1)],
                     "cvssMetricV40": [_metric(_cna, 8.7)]},
         "weaknesses": [{"source": _cna, "type": "Secondary",
                         "description": [{"value": "CWE-79"}, {"value": "CWE-1021"}]},
                        {"source": "nvd@nist.gov", "type": "Primary", "description": [{"value": "CWE-20"}]}]}
_bare = {"id": "CVE-0000-0002", "sourceIdentifier": "cve@mitre.org", "vulnStatus": "Analyzed",
         "metrics": {"cvssMetricV31": [_metric("nvd@nist.gov", 7.5)]},
         "weaknesses": [{"source": "nvd@nist.gov", "type": "Primary",
                         "description": [{"value": "NVD-CWE-noinfo"}]}]}

r1, r2 = flatten(_full), flatten(_bare)
d = to_naive_utc(pd.Series(["2023-01-10T18:15:00.000", "2023-01-10T18:15:00.000Z", "2023-01-10", "not a date"]))
failed = show_conditions({
    "the CNA's v4.0 score beats NVD's and CISA-ADP's v3.1 scores": (r1["cvss_score"], r1["cvss_version"]) == (8.7, "4.0"),
    "NVD's and CISA-ADP's scores kept only as diagnostics": (r1["diag_nvd_score"], r1["diag_adp_score"]) == (9.8, 9.1),
    "no CNA score -> missing flag, not NVD's score": r2["cvss_score"] is None and r2["cvss_missing"],
    "CWEs keep the CNA's order": r1["cwe_primary"] == "CWE-79",
    "NVD's CWE kept out of the CNA list": r1["cwes"] == "CWE-79;CWE-1021" and r1["diag_cwes_non_cna"] == "CWE-20",
    "a placeholder is recorded, not dropped": r2["diag_cwe_placeholders"] == "NVD-CWE-noinfo:NVD",
    "the English description is picked": r1["description"] == "English text",
    "timestamps with and without Z agree": d[0] == d[1],
    "an unreadable date becomes NaT": int(d.isna().sum()) == 1,
    "a same-day pair is 0 days, not -1": int(calendar_days(pd.Series([d[2]]), pd.Series([d[0]])).iloc[0]) == 0,
})
check("A3-parser-selftest", not failed, tests=10, failed=failed)

  ok  the CNA's v4.0 score beats NVD's and CISA-ADP's v3.1 scores
  ok  NVD's and CISA-ADP's scores kept only as diagnostics
  ok  no CNA score -> missing flag, not NVD's score
  ok  CWEs keep the CNA's order
  ok  NVD's CWE kept out of the CNA list
  ok  a placeholder is recorded, not dropped
  ok  the English description is picked
  ok  timestamps with and without Z agree
  ok  an unreadable date becomes NaT
  ok  a same-day pair is 0 days, not -1
PASS  A3-parser-selftest  |  {'tests': 10, 'failed': []}


**Reading the result:** ten `ok` lines and `PASS`. Any `NO` means the parser has changed since
validation; stop and fix it before downloading, because every later table depends on it.

---

## Stage B · Download every NVD record published 2016–2024

### Step 8 · Saving one window safely

**What we are doing:** defining the functions that download one window and keep track of which windows
are finished.

**Why:** the full download makes about 130 requests over 20–30 minutes, and a network drop is likely
at some point. The functions are designed so that an interruption costs at most one window:

- Each window is saved twice: the **raw records** exactly as the NVD sent them (`.jsonl.gz`, so the
  table can be rebuilt later without downloading again) and the **flat rows** from `flatten()`
  (`.parquet`).
- Files are written under temporary `.part` names and renamed only when the window is complete and
  has passed its checks, so a half-written file can never look finished.
- A small manifest (`data/raw/nvd/_manifest.json`) records every finished window, and a re-run
  skips those.

In [15]:
MANIFEST = RAW_NVD / "_manifest.json"
FLAT_COLUMNS = list(flatten({}))          # the parser's column order, also used for an empty window

def window_stem(start, end):
    "File-safe name for a window, e.g. '2016-01-01_2016-04-29' (Windows forbids ':' in file names)."
    return f"{start[:10]}_{end[:10]}"

def load_manifest(path=MANIFEST):
    "Finished windows from earlier runs: {stem: details}. Missing or unreadable -> start afresh."
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def save_manifest(manifest, path=MANIFEST):
    "Write the manifest atomically: to a temporary file, then rename over the old one."
    tmp = path.with_name(path.name + ".part")
    tmp.write_text(json.dumps(manifest, indent=1), encoding="utf-8")
    tmp.replace(path)

def window_done(stem, manifest):
    "A window counts as finished only if the manifest says so and both of its files exist."
    return (stem in manifest and (RAW_NVD / f"{stem}.parquet").exists()
            and (RAW_NVD / f"{stem}.jsonl.gz").exists())

The cell above defines the bookkeeping: file names that are safe on Windows, and a manifest of finished windows that is written atomically, so a crash can't corrupt it.

**Downloading one window:** the next cell defines `window_problems`, the four checks every window must pass, and `download_window`, which fetches a window, checks it, and only then renames its files to their final names.

In [16]:
def window_problems(df, start, end, info):
    "Checks for one downloaded window; returns the list of problems (empty if all is well)."
    lo, hi = pd.Timestamp(start.rstrip("Z")), pd.Timestamp(end.rstrip("Z"))
    published = df["published"] if len(df) else pd.Series(dtype="datetime64[ns]")
    conditions = {
        "rows equal NVD's total for the window": bool(info.get("complete")),
        "every publication date is readable": int(published.isna().sum()) == 0,
        "every record is published inside the window": bool(((published >= lo) & (published < hi)).all()),
        "no CVE appears twice": not (len(df) and df["cve_id"].duplicated().any()),
    }
    return [name for name, passed in conditions.items() if not passed]    # quiet unless it fails

def download_window(client, start, end, folder=RAW_NVD):
    "Download one window; save raw records and flat rows; return a summary dict."
    stem = window_stem(start, end)
    raw_part, pq_part = folder / f"{stem}.jsonl.gz.part", folder / f"{stem}.parquet.part"
    rows = []
    try:
        with gzip.open(raw_part, "wt", encoding="utf-8") as raw:   # raw records, one JSON per line
            for cve in client.iter_cves(pubStartDate=start, pubEndDate=end):
                raw.write(json.dumps(cve) + "\n")
                rows.append(flatten(cve))
    except Exception:
        raw_part.unlink(missing_ok=True)                           # no half-written leftovers
        raise
    info = dict(client.last_run)                                   # expected / received / complete
    df = pd.DataFrame(rows, columns=FLAT_COLUMNS)                  # same columns even if empty
    df["published"] = to_naive_utc(df["published"])
    df["diag_last_modified"] = to_naive_utc(df["diag_last_modified"])
    df["window"] = stem                                            # provenance: which window
    problems = window_problems(df, start, end, info)
    if problems:                                                   # never keep a doubtful window
        raw_part.unlink(missing_ok=True)
        return {**info, "rows": len(df), "problems": problems}
    df.to_parquet(pq_part, index=False)
    raw_part.replace(folder / f"{stem}.jsonl.gz")                  # rename only once complete
    pq_part.replace(folder / f"{stem}.parquet")
    return {**info, "rows": len(df), "rejected": int(df["rejected"].sum()), "problems": [],
            "start": start, "end": end, "saved": now_iso()}

**Reading the result:** no output; this cell only defines functions.

### Step 9 · The download plan

**What we are doing:** listing the 28 windows, marking those already finished by an earlier run, and
estimating how long the rest will take.

**Why:** a plan printed before a long job shows what is about to happen and makes a stalled run easy to
spot. The estimate uses the validation's record count: one request per 2,000 records plus one per
window, each followed by the 6-second pause.

In [17]:
manifest = load_manifest()
pending = [(s, e) for s, e in WINDOWS if not window_done(window_stem(s, e), manifest)]

expected_total = EXPECTED["total_records"] if EXPECTED else 210_000
requests_left = round(expected_total * len(pending) / len(WINDOWS) / NVDClient.PAGE) + len(pending)
print(f"{len(WINDOWS)} windows: {len(WINDOWS) - len(pending)} already finished, {len(pending)} to download")
print(f"about {requests_left} requests, roughly {requests_left * (NVD_PAUSE + 4) / 60:.0f} minutes "
      f"(a {NVD_PAUSE:.0f} s pause plus a few seconds per response)")
if pending:
    print(f"first window to fetch: {window_stem(*pending[0])}")

28 windows: 28 already finished, 0 to download
about 0 requests, roughly 0 minutes (a 6 s pause plus a few seconds per response)


**Reading the result:** on the first run, `0 already finished, 28 to download` and an estimate of
about 130 requests. On a re-run, finished windows are counted and skipped.

### Step 10 · Run the download

**What we are doing:** downloading every pending window in order, printing one line per window.
A window that fails its checks is downloaded once more; if it fails again, it is reported and the
loop moves on.

**Why:** retrying once absorbs a brief glitch (for example, NVD's total changing while a window is
being paged), while moving on keeps one stubborn window from blocking the other 27. Whatever is
still missing is listed at the end, and running this cell again retries only that.

In [18]:
client = NVDClient(NVD_API_KEY)
failures, t0 = {}, time.perf_counter()

for number, (start, end) in enumerate(WINDOWS, 1):
    stem = window_stem(start, end)
    if window_done(stem, manifest):
        continue
    print(f"[{number:2d}/{len(WINDOWS)}] {stem} ...")
    for attempt in range(1, NVD_ATTEMPTS_PER_WINDOW + 1):
        try:
            info = download_window(client, start, end)
        except (ApiError, KeyError, ValueError) as exc:          # network, or an unexpected page
            info = {"complete": False, "problems": [repr(exc)[:200]]}
        if not info["problems"]:
            break
        print(f"  attempt {attempt} failed: {info['problems']}")
    if info["problems"]:
        failures[stem] = info["problems"]
        continue
    manifest[stem] = info
    save_manifest(manifest)                                       # progress survives a crash
    minutes = (time.perf_counter() - t0) / 60
    print(f"  {info['rows']:,} records ({info['rejected']:,} rejected), total {info['expected']:,} "
          f"reconciled | {minutes:.1f} min elapsed")

note_source("NVD CVE API 2.0", windows=len(manifest), endpoint=NVD_URL)
check("B1-nvd-windows", not failures and len(manifest) >= len(WINDOWS),
      finished=len(manifest), of=len(WINDOWS), failed=list(failures))

PASS  B1-nvd-windows  |  {'finished': 28, 'of': 28, 'failed': []}


**Reading the result:** one line per window with its record count, each reconciled against the
NVD's own total, then `PASS  B1-nvd-windows` with `finished: 28`. Windows from 2016 hold about 2,000
records each; recent ones hold about 13,000, so later windows take longer.

**If something fails:** the `FAIL` lists the windows that could not be completed. Run this cell again
(or the whole notebook); finished windows are skipped.

### Step 11 · Combine the windows into one table

**What we are doing:** reading the 28 window files into one table, removing any duplicate CVE, and
checking the result against the validation run's counts.

**Why:** this is the table every later step uses, so it is checked as a whole: the total number of
records, the number of live (non-withdrawn) records, readable dates, and the date range.

In [19]:
def combine_windows(windows=WINDOWS, folder=RAW_NVD):
    "Read every finished window, in order, into one table; drop duplicate CVE ids (keep the first)."
    files = [folder / f"{window_stem(s, e)}.parquet" for s, e in windows]
    present = [f for f in files if f.exists()]
    if not present:
        return pd.DataFrame(), 0, len(files)
    table = pd.concat((pd.read_parquet(f) for f in present), ignore_index=True)
    before = len(table)
    table = table.drop_duplicates("cve_id", keep="first").reset_index(drop=True)
    return table, before - len(table), len(files) - len(present)

nvd, duplicates, missing_files = combine_windows()
if nvd.empty:
    check("B2-nvd-combined", False, error="no window files: fix Step 10 first")
else:
    live_count = int((~nvd["rejected"]).sum())
    print(f"{len(nvd):,} records = {live_count:,} live + {len(nvd) - live_count:,} rejected "
          f"| duplicates removed: {duplicates} | window files missing: {missing_files}")
    failed = show_conditions({
        "all 28 window files present": missing_files == 0,
        "every publication date readable": int(nvd["published"].isna().sum()) == 0,
        f"every record published {FIRST_YEAR}-{LAST_YEAR}": bool(nvd["published"].between(*SPAN, inclusive="left").all()),
    })
    vs_total = compare_to_expected(len(nvd), EXPECTED and EXPECTED["total_records"])
    vs_live = compare_to_expected(live_count, EXPECTED and EXPECTED["live_records"])
    ok = False if failed or vs_total is False or vs_live is False else (True if vs_total and vs_live else None)
    check("B2-nvd-combined", ok, records=len(nvd), live=live_count,
          expected=EXPECTED and EXPECTED["total_records"],
          expected_live=EXPECTED and EXPECTED["live_records"], duplicates=duplicates, failed=failed)
    if RUN["checks"]["B2-nvd-combined"]["verdict"] == "FAIL":                       # never leave an incomplete table where a later notebook reads
        print("not saved: an incomplete table must not be mistaken for the real one. Re-run Step 10.")
    else:
        nvd.to_parquet(INTERIM / "nvd_2016_2024.parquet", index=False)
        RUN["files"]["nvd_2016_2024"] = str(INTERIM / "nvd_2016_2024.parquet")
nvd.head(3)

200,978 records = 187,219 live + 13,759 rejected | duplicates removed: 0 | window files missing: 0
  ok  all 28 window files present
  ok  every publication date readable
  ok  every record published 2016-2024
PASS  B2-nvd-combined  |  {'records': 200978, 'live': 187219, 'expected': 200978, 'expected_live': 187219, 'duplicates': 0, 'failed': []}


,cve_id,published,rejected,cna,description,description_len,cvss_score,cvss_severity,cvss_vector,cvss_version,...,diag_adp_version,diag_score_sources,diag_cwes_non_cna,diag_cwe_placeholders,diag_cwe_entry_types,diag_n_references,diag_ref_tags,diag_n_configurations,diag_has_nvd_kev_fields,window
0,CVE-2015-7441,2016-01-01 00:59:00.153,False,psirt@us.ibm.com,Remote Artifact Loader (RAL) in IBM WebSphere ...,378,None,None,None,None,...,NaN,2.0:NVD;3.0:NVD,CWE-17,,NVD:Primary,8,Vendor Advisory,1,False,2016-01-01_2016-04-29
1,CVE-2015-7489,2016-01-01 00:59:01.057,False,psirt@us.ibm.com,IBM SPSS Statistics 22.0.0.2 before IF10 and 2...,191,None,None,None,None,...,NaN,2.0:NVD;3.0:NVD,CWE-264,,NVD:Primary,4,Vendor Advisory,1,False,2016-01-01_2016-04-29
2,CVE-2015-4941,2016-01-01 05:59:00.097,False,psirt@us.ibm.com,IBM WebSphere MQ Light 1.x before 1.0.2 mishan...,183,None,None,None,None,...,NaN,2.0:NVD;3.0:NVD,CWE-17,,NVD:Primary,6,Vendor Advisory,1,False,2016-01-01_2016-04-29


**Reading the result:** the target is 200,978 records, 187,219 of them live, with no duplicates
(validation step 15 showed that window edges don't overlap). An exact match is `PASS`. A difference
within 0.5% is `WARN`: the NVD can withdraw or add a record between runs, so read the numbers, but
the table is usable. A larger difference is `FAIL` and points at an incomplete download; in that case the
table is **not saved**, so that no later notebook can mistake it for the real one.

---

## Stage C · Snapshot of the KEV catalogue

### Step 12 · Load the pinned snapshot, or download and save a dated copy

**What we are doing:** loading the KEV catalogue version named in `KEV_SNAPSHOT` from its saved copy
or, the first time (or when the setting is `None`), downloading CISA's Known Exploited Vulnerabilities
catalogue and saving it under its own version number. Either way, it is checked as validation step 12
did.

**Why:** KEV supplies the outcome the project predicts, and it changes every few days. Saving the exact
copy used, named by CISA's `catalogVersion`, and **pinning** that version makes the labels
reproducible: a re-run builds them from the same snapshot instead of from whatever CISA published that
morning. (A later catalogue would not change `label_365` for 2021–2024 CVEs, since every 365-day window
has closed, but it would change the KEV dates of CVEs listed late.) Only two columns are ever used later, the CVE ID and the date added; the other
columns exist only for exploited CVEs, so they would hand the answer to a model.

In [20]:
KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

def download_kev(folder=RAW_KEV):
    "Download KEV; save the raw JSON and a parquet copy named by catalogVersion; return (table, meta)."
    raw = http_get(KEV_URL).json()
    meta = {"catalog_version": raw.get("catalogVersion"), "date_released": raw.get("dateReleased"),
            "count": raw.get("count")}
    stem = f"kev_{meta['catalog_version'] or dt.date.today().isoformat()}"
    (folder / f"{stem}.json").write_text(json.dumps(raw), encoding="utf-8")
    (folder / f"{stem}.meta.json").write_text(json.dumps({**meta, "downloaded": now_iso(), "url": KEV_URL}),
                                              encoding="utf-8")       # when and where it came from
    kev = pd.DataFrame(raw["vulnerabilities"])
    kev["cveID"] = kev["cveID"].str.strip().str.upper()             # normalise the join key
    kev["dateAdded"] = to_naive_utc(kev["dateAdded"])
    kev.to_parquet(folder / f"{stem}.parquet", index=False)
    return kev, {**meta, "file": str(folder / f"{stem}.parquet"), "origin": "downloaded"}

The cell above defines `download_kev`, which also writes a small `.meta.json` file recording when the
copy was downloaded. The next cell defines its counterpart.

**Loading a saved snapshot:** `load_kev_snapshot` reads the saved copy of one version. Its download
time comes from the `.meta.json` file or, for a copy saved before that file existed, from the time the
copy was written to disk. That is the access date reported for the source.

In [21]:
def load_kev_snapshot(version, folder=RAW_KEV):
    "The saved snapshot of one KEV version -> (table, meta), or None if its files are missing."
    stem = f"kev_{version}"
    raw_path, table_path, meta_path = (folder / f"{stem}.json", folder / f"{stem}.parquet",
                                       folder / f"{stem}.meta.json")
    if not (raw_path.exists() and table_path.exists()):
        return None
    raw = json.loads(raw_path.read_text(encoding="utf-8"))
    if meta_path.exists():
        downloaded = json.loads(meta_path.read_text(encoding="utf-8"))["downloaded"]
    else:                                        # saved before .meta.json files existed: use the save time
        saved = dt.datetime.fromtimestamp(raw_path.stat().st_mtime).astimezone()
        downloaded = saved.isoformat(timespec="seconds")
    meta = {"catalog_version": raw.get("catalogVersion"), "date_released": raw.get("dateReleased"),
            "count": raw.get("count"), "file": str(table_path), "origin": "pinned snapshot",
            "downloaded": downloaded}
    return pd.read_parquet(table_path), meta

The cell above defines `load_kev_snapshot`; nothing has been loaded yet.

**Load or download, then check:** the next cell loads the pinned version if its files exist, and
downloads the current catalogue otherwise. It then applies the same checks as validation step 12, plus
two more: the number of entries dated to the catalogue's first day must equal the validation run's,
and the version used must be the pinned one.

In [22]:
kev = pd.DataFrame()
try:
    loaded = load_kev_snapshot(KEV_SNAPSHOT) if KEV_SNAPSHOT else None
    kev, kev_meta = loaded if loaded else download_kev()
    note_source("CISA KEV catalog", **kev_meta)
    pinned = KEV_SNAPSHOT is None or kev_meta["catalog_version"] == KEV_SNAPSHOT
    RUN["files"]["kev_snapshot"] = kev_meta["file"]
    first_day = kev["dateAdded"].min()
    launch_day = int((kev["dateAdded"] == first_day).sum())
    print(f"version {kev_meta['catalog_version']} ({kev_meta['origin']}) | released {kev_meta['date_released']}")
    print(f"{len(kev):,} entries | {launch_day} dated to its first day, {first_day:%Y-%m-%d}")
    if not pinned:
        print(f"note: the pinned version {KEV_SNAPSHOT} was not found, so today's catalogue was used")
    failed = show_conditions({
        "rows equal the catalogue's own count": len(kev) == kev_meta["count"],
        "no CVE listed twice": not kev["cveID"].duplicated().any(),
        "every date added is readable": int(kev["dateAdded"].isna().sum()) == 0,
        "Log4Shell is listed (a known answer)": "CVE-2021-44228" in set(kev["cveID"]),
        "the first day is November 3, 2021": first_day == pd.Timestamp(KEV_LAUNCH),
    })
    vs_launch = compare_to_expected(launch_day, EXPECTED and EXPECTED["kev_launch_day_entries"], tolerance=0)
    ok = False if failed or vs_launch is False else (None if not pinned else (vs_launch and True))
    check("C1-kev-snapshot", ok, version=kev_meta["catalog_version"], origin=kev_meta["origin"],
          entries=len(kev), launch_day_entries=launch_day, failed=failed)
except (ApiError, ValueError, KeyError) as exc:
    check("C1-kev-snapshot", False, error=repr(exc))

version 2026.11.10 (pinned snapshot) | released 2026-11-10T19:03:29.8375Z
1,779 entries | 287 dated to its first day, 2021-11-03
  ok  rows equal the catalogue's own count
  ok  no CVE listed twice
  ok  every date added is readable
  ok  Log4Shell is listed (a known answer)
  ok  the first day is November 3, 2021
PASS  C1-kev-snapshot  |  {'version': '2026.11.10', 'origin': 'pinned snapshot', 'entries': 1779, 'launch_day_entries': 287, 'failed': []}


**Reading the result:** version 2026.11.10 (`pinned snapshot`), 1,779 entries, 287 of them dated to the
first day, November 3, 2021. On the very first run the version is `downloaded` instead. `WARN` means
the pinned version's files were missing, so the current catalogue was used: the labels are still
checked, but they are no longer built from the pinned snapshot. The 287 must match the validation run
exactly: entries added on the first day can't change, so a different number would mean CISA revised
old dates. Note the version printed here; it identifies the snapshot behind every label. To move to a newer
catalogue deliberately, set `KEV_SNAPSHOT = None`, run the notebook, and pin the new version it prints.

---

## Stage D · The analysis table

### Step 13 · The label and the split

**What we are doing:** defining the two functions that turn the NVD table and the KEV snapshot into
the analysis table.

**Why:** these few lines decide what counts as *exploited* and which period each CVE belongs to, so they
are written once, as functions, and tested below before being used:

- **Label:** a CVE is exploited (`label_365 = True`) if KEV listed it within 365 calendar days of its
  NVD publication. A CVE listed *before* its publication day counts as exploited and is also flagged
  (`kev_before_publication`), so that every result can be reported with and without those cases.
- **Split:** each CVE belongs to one period by its publication date: `pre-KEV` (before November 3,
  2021, when no fair label exists), `train`, `validate` or `test`.
- **Leakage guard:** only the date added crosses over from KEV, and withdrawn records are dropped.

In [23]:
def assign_split(published, splits=SPLITS):
    "Name of the period each publication date falls in, or 'out of range'."
    conditions = [published.ge(start) & published.lt(end) for start, end in splits.values()]
    return pd.Series(np.select(conditions, list(splits), default="out of range"), index=published.index)

def build_analysis_table(nvd, kev, label_days=LABEL_DAYS):
    "Live CVEs with the KEV date, the 365-day label and the split. Only `dateAdded` comes from KEV."
    kev_dates = kev[["cveID", "dateAdded"]].rename(columns={"cveID": "cve_id", "dateAdded": "kev_date_added"})
    table = (nvd.loc[~nvd["rejected"]]                               # withdrawn records are not CVEs
                .drop(columns="rejected")
                .merge(kev_dates, on="cve_id", how="left", validate="one_to_one"))
    days = calendar_days(table["kev_date_added"], table["published"]).astype("Int64")   # <NA> if not listed
    return table.assign(
        in_kev=table["kev_date_added"].notna(),
        days_to_kev=days,
        label_365=days.le(label_days).fillna(False).astype(bool),        # listed within the window
        kev_before_publication=days.lt(0).fillna(False).astype(bool),
        split=assign_split(table["published"]),
        pub_year=table["published"].dt.year,
    )

**What we are doing next:** testing both functions on a tiny hand-made example whose right answers are
known.

**Why:** an off-by-one in the label (counting day 366, or missing a same-day listing) would shift
every result in the project without any error message. The cases below cover each boundary.

In [24]:
_nvd = pd.DataFrame({
    "cve_id":    ["CVE-A", "CVE-B", "CVE-C", "CVE-D", "CVE-E", "CVE-F", "CVE-G"],
    "published": pd.to_datetime(["2023-03-01 20:15", "2023-03-01 09:00", "2023-03-01 09:00",
                                 "2023-03-01 09:00", "2023-03-01 09:00", "2021-06-01 09:00",
                                 "2024-12-31 23:59"]),
    "rejected":  [False, False, False, False, False, False, True],
})
_kev = pd.DataFrame({"cveID": ["CVE-A", "CVE-B", "CVE-C", "CVE-D", "CVE-F", "CVE-X"],
                     "dateAdded": pd.to_datetime(["2023-03-01", "2024-02-29", "2024-03-01",
                                                  "2023-02-20", "2021-11-03", "2023-01-01"]),
                     "notes": ["exploitation details that must never cross over"] * 6})
_t = build_analysis_table(_nvd, _kev).set_index("cve_id")
failed = show_conditions({
    "listed the same day (evening publication) -> day 0, exploited": (_t.loc["CVE-A", "days_to_kev"], _t.loc["CVE-A", "label_365"]) == (0, True),
    "listed on day 365 -> exploited": (_t.loc["CVE-B", "days_to_kev"], _t.loc["CVE-B", "label_365"]) == (365, True),
    "listed on day 366 -> not exploited": (_t.loc["CVE-C", "days_to_kev"], _t.loc["CVE-C", "label_365"]) == (366, False),
    "listed before publication -> exploited and flagged": bool(_t.loc["CVE-D", "label_365"] and _t.loc["CVE-D", "kev_before_publication"]),
    "never listed -> not exploited, days missing": (not _t.loc["CVE-E", "label_365"]) and pd.isna(_t.loc["CVE-E", "days_to_kev"]),
    "published before KEV's launch -> pre-KEV split": _t.loc["CVE-F", "split"] == "pre-KEV",
    "a 2023 CVE -> validate split": _t.loc["CVE-A", "split"] == "validate",
    "a withdrawn record is dropped": "CVE-G" not in _t.index,
    "a KEV entry not in the NVD table is ignored": "CVE-X" not in _t.index,
    "no KEV column other than the date crosses over": "notes" not in _t.columns,
})
check("D1-label-selftest", not failed, tests=10, failed=failed)

  ok  listed the same day (evening publication) -> day 0, exploited
  ok  listed on day 365 -> exploited
  ok  listed on day 366 -> not exploited
  ok  listed before publication -> exploited and flagged
  ok  never listed -> not exploited, days missing
  ok  published before KEV's launch -> pre-KEV split
  ok  a 2023 CVE -> validate split
  ok  a withdrawn record is dropped
  ok  a KEV entry not in the NVD table is ignored
  ok  no KEV column other than the date crosses over
PASS  D1-label-selftest  |  {'tests': 10, 'failed': []}


**Reading the result:** ten `ok` lines and `PASS`. Note case A: a CVE published at 20:15 and listed the
same day is day 0, because days are counted between calendar dates, not 24-hour periods.

### Step 14 · Build the table and check it against the validation run

**What we are doing:** building the analysis table from the full NVD table and the KEV snapshot, then
comparing the number of live CVEs and exploited CVEs in each period with the validation run.

**Why:** the validation counted these numbers directly from the NVD (validation step 17). Matching
them confirms, end to end, that the download is complete and the label is computed the same way. The
exploited counts concern periods whose 365-day windows have closed, so they should match **exactly**.

In [25]:
table = pd.DataFrame()
if nvd.empty or kev.empty:
    check("D2-analysis-table", False, error="needs the NVD table (Step 11) and the KEV snapshot (Step 12)")
else:
    try:
        table = build_analysis_table(nvd, kev)
        summary = (table.groupby("split", sort=False)
                        .agg(cves=("cve_id", "size"), exploited=("label_365", "sum"),
                             listed_before_publication=("kev_before_publication", "sum"))
                        .reindex(list(SPLITS)))
        summary["rate"] = (summary["exploited"] / summary["cves"]).map("{:.3%}".format)
        if EXPECTED:
            summary["expected_cves"] = summary.index.map(EXPECTED["live_by_split"])
            summary["expected_exploited"] = summary.index.map(EXPECTED["positives_by_split"])
        print(summary.to_string())
        exact = {s: int(summary.loc[s, "exploited"]) == EXPECTED["positives_by_split"].get(s)
                 for s in ("train", "validate", "test")} if EXPECTED else {}
        sizes = [compare_to_expected(int(summary.loc[s, "cves"]), EXPECTED["live_by_split"].get(s))
                 for s in SPLITS] if EXPECTED else [None]
        failed = show_conditions({
            "every CVE falls in a period": int((table["split"] == "out of range").sum()) == 0,
            "one row per CVE": not table["cve_id"].duplicated().any(),
            **{f"{s} exploited count matches the validation run": ok for s, ok in exact.items()},
        })
        ok = False if failed or False in sizes else (True if EXPECTED and all(sizes) else None)
        check("D2-analysis-table", ok, rows=len(table),
              exploited={s: int(summary.loc[s, "exploited"]) for s in SPLITS}, failed=failed)
    except (KeyError, ValueError, pd.errors.MergeError) as exc:
        table = pd.DataFrame()
        check("D2-analysis-table", False, error=repr(exc))

           cves  exploited  listed_before_publication    rate  expected_cves  expected_exploited
split                                                                                           
pre-KEV   90082        159                          0  0.177%          90082                 159
train     28368        128                         35  0.451%          28368                 128
validate  28816        145                         13  0.503%          28816                 145
test      39953        149                          7  0.373%          39953                 149
  ok  every CVE falls in a period
  ok  one row per CVE
  ok  train exploited count matches the validation run
  ok  validate exploited count matches the validation run
  ok  test exploited count matches the validation run
PASS  D2-analysis-table  |  {'rows': 187219, 'exploited': {'pre-KEV': 159, 'train': 128, 'validate': 145, 'test': 149}, 'failed': []}


**Reading the result:** the table should match the validation run:

| Period | Live CVEs | Exploited within 365 days |
|:--|--:|--:|
| pre-KEV (2016 – Nov 2, 2021) | 90,082 | 159 (description only) |
| train (Nov 3, 2021 – 2022) | 28,368 | 128 |
| validate (2023) | 28,816 | 145 |
| test (2024) | 39,953 | 149 |

The live counts may drift by a record or two if the NVD withdrew a record since November 10; that is a
`WARN`. The exploited counts may not drift at all. `listed_before_publication` should show about 35
in the training period, the cases the proposal reports with and without.

### Step 15 · Describe every column, guard against leakage, and save

**What we are doing:** giving every column a role (identifier, model input, label, diagnostic) and a
one-line description, checking that no column is left without one, and saving the table with its
column dictionary.

**Why:** the later notebooks decide what the model may see by reading this dictionary, not by
remembering rules. A new or unexpected column fails the check instead of slipping into the model.
Columns starting with `diag_`, and every label column, are marked as never being model inputs.

In [26]:
COLUMN_ROLES = {
    "cve_id":            ("id",        "CVE identifier"),
    "published":         ("time",      "NVD publication time, UTC; defines the split"),
    "pub_year":          ("time",      "Year of publication"),
    "split":             ("split",     "pre-KEV, train, validate or test, by publication date"),
    "window":            ("provenance", "The download window the record came from"),
    "cna":               ("input",     "The CNA that published the CVE"),
    "description":       ("input",     "English description at download time (see the report's caveat)"),
    "description_len":   ("input",     "Length of the description in characters"),
    "cvss_score":        ("input",     "CVSS base score supplied by the CNA; missing if none"),
    "cvss_severity":     ("input",     "CVSS severity band supplied by the CNA"),
    "cvss_vector":       ("input",     "CVSS vector supplied by the CNA"),
    "cvss_version":      ("input",     "CVSS version of the CNA's score"),
    "cvss_missing":      ("input",     "True if the CNA supplied no CVSS score"),
    "cwes":              ("input",     "CWE ids supplied by the CNA, in the CNA's order"),
    "cwe_primary":       ("input",     "The CNA's first CWE"),
    "cwe_missing":       ("input",     "True if the CNA supplied no CWE"),
    "kev_date_added":    ("label",     "Date CISA added the CVE to KEV (snapshot in data/raw/kev)"),
    "in_kev":            ("label",     "Listed in KEV at any time"),
    "days_to_kev":       ("label",     "Calendar days from publication to KEV listing"),
    "label_365":         ("label",     "OUTCOME: listed in KEV within 365 calendar days of publication"),
    "kev_before_publication": ("label", "Listed in KEV before the publication day"),
}
DIAGNOSTIC = ("diagnostic", "Added after publication or part of NVD's workflow: never a model input")

def column_dictionary(table):
    "One row per column: name, role, may_be_model_input, description. Unknown columns -> role None."
    rows = []
    for column in table.columns:
        role, text = COLUMN_ROLES.get(column) or (DIAGNOSTIC if column.startswith("diag_") else (None, None))
        rows.append({"column": column, "role": role, "may_be_model_input": role == "input",
                     "dtype": str(table[column].dtype), "description": text})
    return pd.DataFrame(rows)

The cell above lists the role of every named column, and treats any column starting with `diag_` as diagnostic.

**Check and save:** the next cell builds the column dictionary, fails if any column has no role or if a label or diagnostic column is marked as an input, and saves the table and the dictionary.

In [27]:
if table.empty or RUN["checks"].get("D2-analysis-table", {}).get("verdict") == "FAIL":
    check("D3-table-saved", False, error="not saved: Step 14 failed, and an incomplete table must not be "
                                         "mistaken for the real one")
else:
    columns = column_dictionary(table)
    unknown = columns.loc[columns["role"].isna(), "column"].tolist()
    inputs = columns.loc[columns["may_be_model_input"], "column"].tolist()
    print(columns.groupby("role")["column"].count().to_string())
    print(f"\npossible model inputs ({len(inputs)}): {inputs}")
    failed = show_conditions({
        "every column has a role": not unknown,
        "no label or diagnostic column is marked as an input":
            not any(c.startswith("diag_") or COLUMN_ROLES.get(c, ("",))[0] == "label" for c in inputs),
        # Independent of the roles above: an input must be a column the NVD parser produces, so
        # nothing derived from KEV can become an input, even if its role were mislabelled.
        "every input comes from the NVD parser, none from KEV": set(inputs) <= set(FLAT_COLUMNS),
    })
    if failed:                                  # a table whose dictionary could let a label in is not saved
        print("not saved: fix COLUMN_ROLES above, then run this cell again")
    else:
        table.to_parquet(PROCESSED / "analysis_table.parquet", index=False)
        columns.to_csv(PROCESSED / "analysis_table_columns.csv", index=False)
        RUN["files"].update(analysis_table=str(PROCESSED / "analysis_table.parquet"),
                            column_dictionary=str(PROCESSED / "analysis_table_columns.csv"))
    check("D3-table-saved", not failed, rows=len(table), columns=len(columns), unknown=unknown, failed=failed)

role
diagnostic    14
id             1
input         11
label          5
provenance     1
split          1
time           2

possible model inputs (11): ['cna', 'description', 'description_len', 'cvss_score', 'cvss_severity', 'cvss_vector', 'cvss_version', 'cvss_missing', 'cwes', 'cwe_primary', 'cwe_missing']
  ok  every column has a role
  ok  no label or diagnostic column is marked as an input
  ok  every input comes from the NVD parser, none from KEV
PASS  D3-table-saved  |  {'rows': 187219, 'columns': 35, 'unknown': [], 'failed': []}


**Reading the result:** a count of columns per role, then the 11 columns a model may use: the
publishing CNA, the description and its length, and the CNA's own severity and weakness fields.
`PASS` means every column is accounted for and no label can reach a model.

The table is **not saved** if this check fails, or if Step 14 failed, so that later notebooks never
start from an incomplete table or one whose dictionary could let a label in. Two caveats are recorded
for the report rather than fixed here:

- The **description** is the current text, not necessarily the text at publication. Validation step 16
  found that descriptions are occasionally edited; the change history can rebuild the original text if
  the analysis shows that matters.
- A **CNA's score** can also be added after publication, though rarely; the same history can check this.

---

## Stage E · The EPSS benchmark

### Step 16 · Reading one day's EPSS file

**What we are doing:** defining the functions that download one of FIRST's daily EPSS files and keep
only the rows we need. They are the functions that passed validation step 13b.

**Why:** question Q4 compares our model with EPSS *as it stood shortly after each CVE was published*,
not today, because today's scores already reflect the exploitation we are trying to predict. FIRST
publishes one file per day containing every score; validation step 13b showed that these files agree
exactly with the API and name the model that produced them. One file per day is far faster than asking
the API for each CVE and date separately.

In [28]:
EPSS_FILE_HOSTS = ["https://epss.empiricalsecurity.com",   # current host, per FIRST's data page
                   "https://epss.cyentia.com"]             # older host, tried only as a fallback

def parse_epss_header(line):
    "'#model_version:v2023.03.01,score_date:2024-01-02T00:00:00+0000' -> dict (parsed loosely)."
    fields = {}
    for part in (line or "").lstrip("#").split(","):
        key, sep, value = part.partition(":")
        if sep:
            fields[key.strip()] = value.strip()
    return fields

def epss_daily_file(day):
    "One day's complete EPSS file -> (scores, header fields, url). Raises ApiError if no host has it."
    name, problems = f"epss_scores-{day}.csv.gz", []
    for host in EPSS_FILE_HOSTS:
        try:
            resp = http_get(f"{host}/{name}", headers={"Accept": "*/*"}, max_attempts=2,
                            pause=EPSS_PAUSE, timeout=120)
        except ApiError as exc:
            problems.append(f"{host}: {exc}")
            continue
        raw = resp.content
        # A server may send the file already decompressed; gzip files start with bytes 1f 8b.
        text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8")
        first, _, rest = text.partition("\n")
        header = first.strip() if first.startswith("#") else ""
        scores = pd.read_csv(io.StringIO(rest if header else text), dtype={"cve": str})
        scores["cve"] = scores["cve"].str.strip().str.upper()
        return scores, parse_epss_header(header), f"{host}/{name}"
    raise ApiError(f"no host served {name}: " + " | ".join(problems))

**Reading the result:** no output; this cell only defines functions.

### Step 17 · Which scores are needed, from which days

**What we are doing:** listing, for every live 2024 CVE, the three days whose scores we need: 1, 7 and
30 days after its publication day. Because a brand-new CVE is sometimes missing from the file of its
target day, the next two days are also kept as fallbacks. From that list we work out which CVEs to keep
from each day's file.

**Why:** a daily file holds every scored CVE (about 220,000 in 2024), but on any given day we need only
a few hundred of them. Keeping just those rows makes the saved files tiny, and the plan below tells us
exactly which days to download.

In [29]:
def benchmark_targets(table, split=EPSS_SPLIT, horizons=EPSS_HORIZONS):
    "One row per (CVE, horizon): the day whose score is wanted."
    cves = table.loc[table["split"] == split, ["cve_id", "published"]]
    targets = cves.assign(pub_day=cves["published"].dt.normalize()).merge(
        pd.DataFrame({"horizon_days": list(horizons)}), how="cross")
    targets["target_day"] = targets["pub_day"] + pd.to_timedelta(targets["horizon_days"], unit="D")
    return targets.drop(columns="published")

def needed_by_day(targets, slack=EPSS_SLACK_DAYS):
    "{day 'YYYY-MM-DD': set of CVE ids to keep from that day's file}, fallback days included."
    candidate = targets[["cve_id", "target_day"]].merge(pd.DataFrame({"offset": range(slack + 1)}), how="cross")
    candidate["day"] = (candidate["target_day"] + pd.to_timedelta(candidate["offset"], unit="D")).dt.strftime("%Y-%m-%d")
    return candidate.groupby("day")["cve_id"].agg(set).to_dict()

table_ok = not table.empty and RUN["checks"].get("D2-analysis-table", {}).get("verdict") != "FAIL"
targets = benchmark_targets(table) if table_ok else pd.DataFrame()    # only from a complete table
NEEDED = needed_by_day(targets) if len(targets) else {}
if NEEDED:
    days = sorted(NEEDED)
    print(f"{targets['cve_id'].nunique():,} {EPSS_SPLIT} CVEs x {len(EPSS_HORIZONS)} horizons = {len(targets):,} scores wanted")
    print(f"{len(days)} daily files needed, {days[0]} to {days[-1]}; "
          f"about {np.mean([len(v) for v in NEEDED.values()]):,.0f} CVEs kept per day")
else:
    print("no benchmark plan: the analysis table is missing or failed Step 14, so fix Stage D first")

39,953 test CVEs x 3 horizons = 119,859 scores wanted
397 daily files needed, 2024-01-02 to 2025-02-01; about 906 CVEs kept per day


**Reading the result:** with the validation's count of 39,953 live test CVEs, that is 119,859 scores
wanted, drawn from 397 daily files running from January 2, 2024 to February 1, 2025.

### Step 18 · Download the daily files

**What we are doing:** downloading each needed day's file, keeping only that day's needed rows, and
saving them as a small parquet file together with the model version named in the file's header. Days
already saved by an earlier run are skipped.

**Why:** saving each day as it arrives makes the download resumable, like the NVD windows. Recording the
model version on every row lets the report state exactly which EPSS model the benchmark comes from. A
day that no host can supply is reported, and the fallback days usually cover for it.

In [30]:
def save_epss_day(day, wanted, folder=RAW_EPSS):
    "Download one day, keep the wanted CVEs, save them; return a one-line summary dict."
    scores, header, url = epss_daily_file(day)
    kept = scores.loc[scores["cve"].isin(wanted), ["cve", "epss", "percentile"]].rename(columns={"cve": "cve_id"})
    kept = kept.assign(score_day=day, model_version=header.get("model_version"))
    part = folder / f"epss_{day}.parquet.part"
    kept.to_parquet(part, index=False)
    part.replace(folder / f"epss_{day}.parquet")                  # rename only once complete
    return {"day": day, "file_rows": len(scores), "kept": len(kept), "wanted": len(wanted),
            "model_version": header.get("model_version"), "url": url}

epss_log, epss_missing, t0 = [], [], time.perf_counter()
for number, day in enumerate(sorted(NEEDED), 1):
    if (RAW_EPSS / f"epss_{day}.parquet").exists():
        continue                                                   # saved by an earlier run
    try:
        epss_log.append(save_epss_day(day, NEEDED[day]))
    except (ApiError, ValueError, KeyError, OSError) as exc:      # OSError: a corrupt gzip
        epss_missing.append(day)
        print(f"  {day}: not available ({str(exc)[:120]})")
    if number % 25 == 0 or number == len(NEEDED):
        print(f"[{number:3d}/{len(NEEDED)}] up to {day} | {(time.perf_counter() - t0) / 60:.1f} min")

note_source("EPSS daily files (FIRST)", days_downloaded=len(epss_log), days_missing=epss_missing)
saved_days = sum((RAW_EPSS / f"epss_{d}.parquet").exists() for d in NEEDED)
if not NEEDED:
    check("E1-epss-days", False, error="nothing to download: fix Stage D first")
else:
    check("E1-epss-days", True if saved_days == len(NEEDED) else (None if saved_days else False),
          saved=saved_days, of=len(NEEDED), missing=epss_missing)

  HTTP 403: <?xml version="1.0" encoding="UTF-8"?>
<Error><Code>AccessDenied</Code><Message>Access Denied</Message></Error>; waiting 6s (attempt 1/2)
  HTTP 403: <?xml version="1.0" encoding="UTF-8"?>
<Error><Code>AccessDenied</Code><Message>Access Denied</Message></Error>; waiting 6s (attempt 1/2)
  2024-12-01: not available (no host served epss_scores-2024-12-01.csv.gz: https://epss.empiricalsecurity.com: gave up after 2 attempts; last problem)
WARN  E1-epss-days  |  {'saved': 396, 'of': 397, 'missing': ['2024-12-01']}


**Reading the result:** a progress line every 25 days, then `PASS  E1-epss-days` with every day saved.
A few missing days give `WARN`: the fallback days can still supply those scores, and Step 19 measures
whether they did. Running the cell again retries only the missing days.

### Step 19 · Assemble the benchmark table

**What we are doing:** for every test CVE and horizon, taking the score from the target day, or from the
first fallback day that has it, and saving one table: one row per CVE and horizon, with the score, the
day it came from, how many days late that was, and the model version.

**Why:** this is the table Q4 compares against. Recording the actual score day keeps the fallback
visible: a score taken a day late is marked as such, not silently treated as on time.

In [31]:
def assemble_benchmark(targets, folder=RAW_EPSS, slack=EPSS_SLACK_DAYS):
    "One row per (CVE, horizon): the earliest score on or up to `slack` days after the target day."
    files = sorted(folder.glob("epss_*.parquet"))
    if not files:
        return pd.DataFrame()
    scores = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
    scores["score_day"] = pd.to_datetime(scores["score_day"])
    found = targets.merge(scores, on="cve_id", how="inner")
    found["days_late"] = (found["score_day"] - found["target_day"]).dt.days
    found = (found[found["days_late"].between(0, slack)]
                  .sort_values("days_late")
                  .drop_duplicates(["cve_id", "horizon_days"], keep="first"))
    columns = ["cve_id", "horizon_days", "score_day", "days_late", "epss", "percentile", "model_version"]
    return (targets.merge(found[columns], on=["cve_id", "horizon_days"], how="left")
                   .sort_values(["cve_id", "horizon_days"]).reset_index(drop=True))

The cell above defines `assemble_benchmark`; it reads the saved day files but downloads nothing.

**Assemble, check and save:** the next cell builds the benchmark, prints the share of scores found for each horizon and the model versions involved, and saves the table.

In [32]:
benchmark = assemble_benchmark(targets) if len(targets) else pd.DataFrame()
if benchmark.empty:
    check("E2-epss-benchmark", False, error="no EPSS rows: fix Steps 17-18 first")
else:
    coverage = benchmark.groupby("horizon_days")["epss"].apply(lambda s: s.notna().mean())
    late = benchmark.groupby("horizon_days")["days_late"].apply(lambda s: (s > 0).mean())
    versions = {str(k): int(v) for k, v in benchmark["model_version"].dropna().value_counts().items()}
    print(pd.DataFrame({"scored": coverage.map("{:.2%}".format),
                        "taken_from_a_fallback_day": late.map("{:.2%}".format)}).to_string())
    print("\nmodel versions:", versions)
    benchmark.to_parquet(PROCESSED / "epss_benchmark.parquet", index=False)
    RUN["files"]["epss_benchmark"] = str(PROCESSED / "epss_benchmark.parquet")
    worst = float(coverage.min())
    check("E2-epss-benchmark", True if worst >= 0.98 and len(versions) == 1 else (None if worst >= 0.80 else False),
          rows=len(benchmark), coverage={int(h): round(float(c), 4) for h, c in coverage.items()},
          model_versions=versions)

              scored taken_from_a_fallback_day
horizon_days                                  
1             91.96%                     2.56%
7             98.34%                     1.48%
30            98.67%                     0.90%

model versions: {'v2023.03.01': 115452}
WARN  E2-epss-benchmark  |  {'rows': 119859, 'coverage': {1: 0.9196, 7: 0.9834, 30: 0.9867}, 'model_versions': {'v2023.03.01': 115452}}


**Reading the result:** the share of scores found for each horizon, and how many came from a fallback
day. Validation step 13b showed that the whole 2024 test year was scored by one model,
`v2023.03.01`, so a single version is expected. `PASS` needs at least 98% coverage for every horizon
and one model version. Lower coverage, or two versions, is a `WARN` to describe in the report, not a
reason to stop.

---

## Stage F · Summary

### Step 20 · Fingerprints: are the tables the same as last time?

**What we are doing:** computing two SHA-256 fingerprints of every table this notebook saved, and
comparing them with the fingerprints recorded by the previous run (read from the run log before this
run replaces it).

- The **content fingerprint** depends only on the table's column names and values, computed with
  `pd.util.hash_pandas_object` after putting the table in one standard form (dates in microseconds,
  lists as text). Tested on this project's tables, it gives the same answer under pandas 2.3 and 3.0,
  so it is the one compared between runs. (The column dictionary is the exception: it *records* each
  column's data type, and pandas 3 names some types differently, e.g. `str` instead of `object`.)
- The **file fingerprint** is the hash of the file's bytes. It identifies the exact file, but can change
  with the pandas or pyarrow version even when the content is identical, so it is recorded, not compared.

**Why:** reproducibility should be checked, not assumed. A re-run in the same folder must give the
same content (`PASS`). A difference is a `WARN` that names the tables that changed, for example after
deliberately moving to a newer KEV catalogue. A table that was not saved at all is a `FAIL`.

In [33]:
FINGERPRINTED = {"analysis_table": PROCESSED / "analysis_table.parquet",
                 "column_dictionary": PROCESSED / "analysis_table_columns.csv",
                 "epss_benchmark": PROCESSED / "epss_benchmark.parquet",
                 "nvd_2016_2024": INTERIM / "nvd_2016_2024.parquet"}

def hashable(df):
    "A copy in one standard form: list cells (KEV's `cwes`) as JSON text, dates in microseconds."
    out = df.copy()
    for column in out.columns[out.dtypes == object]:
        if out[column].map(lambda v: isinstance(v, (list, tuple, dict, np.ndarray))).any():
            out[column] = out[column].map(lambda v: json.dumps(v.tolist() if isinstance(v, np.ndarray) else v,
                                                               default=str))
    for column in out.columns:                   # pandas versions differ in how finely they store dates
        if pd.api.types.is_datetime64_any_dtype(out[column]):
            out[column] = out[column].astype("datetime64[us]")
    return out

def fingerprint(path):
    "Rows, and SHA-256 fingerprints of a saved table's file bytes and of its content alone."
    path = Path(path)
    df = hashable(pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path))
    content = hashlib.sha256("\x1f".join(map(str, df.columns)).encode("utf-8"))      # column names...
    content.update(pd.util.hash_pandas_object(df, index=False).to_numpy().tobytes())  # ...and every value
    return {"rows": len(df), "content_sha256": content.hexdigest(),
            "file_sha256": hashlib.sha256(path.read_bytes()).hexdigest()}

def previous_fingerprints(path=LOGS / "acquisition_log.json"):
    "(fingerprints, start time) recorded by the previous run, or ({}, None) if it recorded none."
    try:
        log = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}, None
    fingerprints = log.get("fingerprints", {})
    return fingerprints, (log.get("started") if fingerprints else None)   # a run without any: nothing to compare

The cell above defines the two fingerprints and reads the previous run's. (`hashable` first turns cells that hold a list, such as the CWE list in KEV's own table, into text, because a list can't be hashed directly, and stores every date in microseconds, because pandas versions differ on that.) The next cell computes this
run's, prints them beside the previous ones, and records the verdict.

In [34]:
before, before_started = previous_fingerprints()
to_fingerprint = dict(FINGERPRINTED)
if "kev_snapshot" in RUN["files"]:
    to_fingerprint["kev_snapshot"] = Path(RUN["files"]["kev_snapshot"])
RUN["fingerprints"] = {name: fingerprint(path) for name, path in to_fingerprint.items() if path.exists()}
missing = [name for name in to_fingerprint if name not in RUN["fingerprints"]]
changed = [name for name, fp in RUN["fingerprints"].items()
           if name in before and before[name]["content_sha256"] != fp["content_sha256"]]

print(f"compared with the run started {before_started}\n" if before_started
      else "no earlier run recorded fingerprints: these are the first, and the next run will be compared with them\n")
for name, fp in RUN["fingerprints"].items():
    status = "new" if name not in before else ("CHANGED" if name in changed else "same")
    print(f"  {name:<18} {fp['rows']:>9,} rows   content {fp['content_sha256'][:16]}…   {status}")
for name in missing:
    print(f"  {name:<18} not saved by this run")
check("F1-data-fingerprints", False if missing else (None if changed else True),
      compared_with=before_started, changed=changed, missing=missing)

compared with the run started 2026-11-12T22:35:14-08:00

  analysis_table       187,219 rows   content 2979fb727e87c33a…   same
  column_dictionary         35 rows   content 2ccf6a5ba79265fa…   same
  epss_benchmark       119,859 rows   content 94a2fa68080eb9a5…   same
  nvd_2016_2024        200,978 rows   content b067df2bab983a92…   same
  kev_snapshot           1,779 rows   content 777cbc8f44e2729c…   same
PASS  F1-data-fingerprints  |  {'compared_with': '2026-11-12T22:35:14-08:00', 'changed': [], 'missing': []}


### Step 21 · Every check, every file, and the run log

**What we are doing:** listing every check with its verdict, every file written with its size, and the
access time of each source, then saving all of it as `logs/acquisition_log.json`, plus a copy named by the run's start time
(for example `acquisition_log_2026-11-12T2235.json`), so every run's log is kept.

**Why:** access dates for each source are a graded requirement, and a machine-readable log lets the run
be checked later without screenshots. A check that never ran (because a cell was skipped or an
earlier step failed) is listed as `FAIL`, so a partial run can't look complete.

In [35]:
EXPECTED_CHECKS = ["A1-key-loaded", "A2-windows", "A3-parser-selftest", "B1-nvd-windows",
                   "B2-nvd-combined", "C1-kev-snapshot", "D1-label-selftest", "D2-analysis-table",
                   "D3-table-saved", "E1-epss-days", "E2-epss-benchmark",
                   "F1-data-fingerprints"]

def write_run_log(run=RUN, logs=LOGS):
    "Mark checks that never ran as FAIL, add file sizes, and save the log (latest + dated copy)."
    for name in EXPECTED_CHECKS:
        run["checks"].setdefault(name, {"verdict": "FAIL", "error": "check never ran"})
    run["file_sizes_mb"] = {k: round(Path(p).stat().st_size / 1e6, 2)
                            for k, p in run["files"].items() if Path(p).exists()}
    run["finished"] = now_iso()
    text = json.dumps(run, indent=2, default=str)                  # dates and numpy values as text
    stamp = run["started"][:16].replace(":", "")                 # e.g. 2026-11-12T2235: one copy per run
    for path in (logs / "acquisition_log.json", logs / f"acquisition_log_{stamp}.json"):
        path.write_text(text, encoding="utf-8")
    return logs / "acquisition_log.json"

log_path = write_run_log()
tally = Counter(c["verdict"] for c in RUN["checks"].values())
print(f"{tally['PASS']}/{len(RUN['checks'])} checks passed, {tally['WARN']} to review, {tally['FAIL']} failed\n")
for name, c in RUN["checks"].items():
    why = c.get("error") or (c.get("failed") if c["verdict"] != "PASS" else "")
    print(f"  {c['verdict']:<5} {name}" + (f"   <- {why}" if why else ""))
print("\nfiles written:")
for name, size in RUN["file_sizes_mb"].items():
    print(f"  {name:<18} {size:>8.2f} MB   {RUN['files'][name]}")
print("\naccess dates:")
for name, s in RUN["sources"].items():
    print(f"  {name:<26} {s['accessed']}")
print(f"\nrun log: {log_path}")

10/12 checks passed, 2 to review, 0 failed

  PASS  A1-key-loaded
  PASS  A2-windows
  PASS  A3-parser-selftest
  PASS  B1-nvd-windows
  PASS  B2-nvd-combined
  PASS  C1-kev-snapshot
  PASS  D1-label-selftest
  PASS  D2-analysis-table
  PASS  D3-table-saved
  WARN  E1-epss-days
  WARN  E2-epss-benchmark
  PASS  F1-data-fingerprints

files written:
  nvd_2016_2024         29.37 MB   data\interim\nvd_2016_2024.parquet
  kev_snapshot           0.29 MB   data\raw\kev\kev_2026.11.10.parquet
  analysis_table        28.80 MB   data\processed\analysis_table.parquet
  column_dictionary      0.00 MB   data\processed\analysis_table_columns.csv
  epss_benchmark         1.04 MB   data\processed\epss_benchmark.parquet

access dates:
  NVD CVE API 2.0            2026-11-12T22:53:28-08:00
  CISA KEV catalog           2026-11-12T22:53:31-08:00
  EPSS daily files (FIRST)   2026-11-12T22:53:49-08:00

run log: logs\acquisition_log.json


**Reading the result:** the target is 12 of 12 checks passed. On the November 10 data, `E1` and `E2` are `WARN` for the reasons explained in Steps 18 and 19. Any `WARN` should be read and, if it
stands, described in the report; any `FAIL` names the step to look at.

## What comes next

| Notebook | Starts from | Purpose |
|:--|:--|:--|
| `02_descriptive_analysis.ipynb` | `analysis_table.parquet` | Q1 and Q3: how rare exploitation is, where it concentrates, and how fast listing follows publication |
| `03_model.ipynb` | `analysis_table.parquet`, `analysis_table_columns.csv` | Q2: the ranking model, trained on `train`, tuned on `validate`, evaluated once on `test` |
| `04_epss_comparison.ipynb` | `epss_benchmark.parquet` and the model's test scores | Q4: the comparison with EPSS |

Keep this notebook in the repository as `01_acquire_data.ipynb`. With `00_validate_sources.ipynb`, it
is the evidence for the data-acquisition section of the final report.